# 03 · Training the Multimodal Energy Transformer

Trains the full four-stream model on the leakage-safe Ausgrid subgraph dataset.

**Runs on Google Colab.** Select *Runtime → Change runtime type → GPU* for the real run. The
notebook first executes a **CPU smoke test** on a tiny subsample to confirm the whole graph
plumbing works end-to-end, then launches the full training run.

> **Integrity note.** Loss curves printed here are the real training trajectory. Do not paste
> placeholder or hand-made numbers into the paper — only what this notebook actually prints.

In [ ]:
import os, sys, pickle, json
from pathlib import Path
import numpy as np, torch

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # assumes the repo was cloned in notebook 01; re-clone if running fresh
    if not Path('src').exists():
        !git clone https://github.com/<your-user>/paper1-urban-energy-forecasting.git
        %cd paper1-urban-energy-forecasting
        !pip -q install -r requirements.txt
    %cd ..
sys.path.insert(0, os.getcwd())

from src.utils import load_config, smoke_test_config, set_seed
from src.train import train_model
from src.evaluate import evaluate_model, print_table, save_results

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 1 · Load the data module

Reuse the pipeline saved by notebook 01. If it isn't present, rebuild it here (this re-downloads
weather via the Open-Meteo cache).

In [ ]:
cfg = load_config('configs/default.yaml')
pkl = Path(cfg['data']['processed_dir']) / 'dataset.pkl'
if pkl.exists():
    with open(pkl, 'rb') as f:
        dm = pickle.load(f)
    print('loaded cached data module')
else:
    from src.data.dataset import AusgridDataModule
    dm = AusgridDataModule(cfg); dm.setup()
    print('rebuilt data module from source')

print('nodes=%d | train=%d | test=%d | K=%d' % (
    dm.train_ds.n_nodes, len(dm.train_ds), len(dm.test_ds), dm.train_ds.K))

## 2 · CPU smoke test (tiny subsample)

Confirms the full forward/backward path — building + weather + temporal + GAT + cross-attention +
multi-task heads — executes without shape errors, on a small slice and 2 epochs. This is a
**correctness check, not a result**: its loss numbers are meaningless and are never reported.

> Note: run this with the full `dm` for the graph, but reduced `epochs`/`d_model` via
> `smoke_test_config`. On Colab a full-scale smoke test still runs in well under a minute on CPU.

In [ ]:
smoke = smoke_test_config(cfg)   # small d_model, 1 layer, 2 epochs, fewer neighbours
# For a genuine speed-up, also cap the number of customers used. If your data module
# supports a subsample, set cfg['data']['n_customers_limit'] before setup(); otherwise the
# small model itself is fast enough on CPU for the smoke test.
print('smoke config: d_model=%d, layers=%d, epochs=%d, K=%d' % (
    smoke['model']['d_model'], smoke['model']['n_layers'],
    smoke['training']['epochs'], smoke['data']['max_neighbors']))

smoke_model, smoke_hist = train_model(smoke, dm, device='cpu', out_dir='runs/smoke')
print('smoke finished. final train loss = %.4f (NOT a result — correctness check only)' % smoke_hist['train_loss'][-1])

## 3 · Full training run

Default config: `d_model=128`, 3 transformer layers, 4 GAT heads, patch length 24 h / stride 12 h,
168-h lookback, 4 horizons (1 h, 6 h, 24 h, 168 h), weighted multi-task MSE (0.6 demand / 0.4 gen),
AdamW + cosine LR, early stopping on a chronological 15% validation slice carved from the **train**
window (never the test period).

In [ ]:
model, history = train_model(cfg, dm, device=device, out_dir='runs/base')

import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.plot(history['train_loss'], label='train loss')
plt.plot(history['val_loss'], label='val loss')
plt.xlabel('epoch'); plt.ylabel('weighted multi-task MSE'); plt.legend()
plt.title('Training trajectory (real run)'); plt.tight_layout()
plt.savefig('runs/base/loss_curve.png', bbox_inches='tight'); plt.show()
print('best val loss: %.4f' % min(history['val_loss']))

## 4 · Ablations

Three architectural ablations isolate the contribution of each stream:

| run | what changes | tests |
|-----|--------------|-------|
| `no_graph` | cross-neighbour edges masked; spatial stream sees only the root | does the GAT add value? |
| `single_task` | generation loss weight set to 0 (head stays, receives no gradient); demand-only loss | does joint learning help demand? |
| `no_cross_attn` | streams concatenated instead of cross-attended | does fusion matter? |

`no_graph` is wired via `graph_ablation='no_graph'` in `train_model`. The others are config
toggles; run each, then compare in notebook 04.

In [ ]:
# no-graph ablation
model_noG, hist_noG = train_model(cfg, dm, device=device,
                                  out_dir='runs/no_graph', graph_ablation='no_graph')
print('no_graph best val loss: %.4f' % min(hist_noG['val_loss']))

In [ ]:
# single-task (demand only) ablation: set gen_weight=0 so the gen head
# receives no gradient signal; demands its prediction is ignored downstream.
import copy
cfg_st = copy.deepcopy(cfg)
cfg_st['training']['demand_weight'] = 1.0
cfg_st['training']['gen_weight'] = 0.0
model_st, hist_st = train_model(cfg_st, dm, device=device, out_dir='runs/single_task')
print('single_task best val loss: %.4f' % min(hist_st['val_loss']))

In [ ]:
# no-cross-attention ablation: streams are concatenated, not cross-attended.
import copy
cfg_nca = copy.deepcopy(cfg)
cfg_nca['model']['no_cross_attn'] = True
model_nca, hist_nca = train_model(cfg_nca, dm, device=device, out_dir='runs/no_cross_attn')
print('no_cross_attn best val loss: %.4f' % min(hist_nca['val_loss']))

## 5 · Quick test-set check (headline numbers come from notebook 04)

Evaluate the full model on the held-out test year to confirm the pipeline is sound. The full
baseline + ablation comparison is notebook 04.

In [ ]:
results = evaluate_model(model, dm, cfg, device, split='test')
print_table(results)
save_results(results, 'runs/base/test_metrics.json')